# Task 1: Install required Python libraries

In [ ]:
%pip install -q pandas scikit-learn streamlit joblib requests

import pandas
import sklearn
import streamlit
import joblib
import requests

print(pandas.__version__)
print(sklearn.__version__)
print(streamlit.__version__)
print(joblib.__version__)
print(requests.__version__)

# Task 2: Download the dataset

In [ ]:
import pandas as pd
import requests
from pathlib import Path

# Create data directory if it doesn't exist
Path("data").mkdir(exist_ok=True)

csv_path = Path("data/results.csv")

# Download only if the file doesn't already exist
if not csv_path.exists():
    url = "https://raw.githubusercontent.com/academic-initiative/skillsbuild/main/ai-in-sports/football-predictor/data/results.csv"
    response = requests.get(url)
    response.raise_for_status()
    csv_path.write_bytes(response.content)
    print("Downloaded results.csv")
else:
    print("results.csv already exists, skipping download.")

# Load into DataFrame
matches = pd.read_csv(csv_path)
matches["date"] = pd.to_datetime(matches["date"])

print("Shape:", matches.shape)
print("Date range:", matches["date"].min(), "→", matches["date"].max())
matches.head(3)

# Task 3: Explore the data

In [ ]:
# --- Top 10 most frequent tournaments ---
print("Top 10 tournaments by match count:")
print(matches["tournament"].value_counts().head(10).to_string())

# --- Top 15 teams by total matches played ---
print("\nTop 15 teams by total matches played:")
team_counts = (
    pd.concat([matches["home_team"], matches["away_team"]])
    .value_counts()
    .head(15)
)
print(team_counts.to_string())

# --- Matches per decade ---
print("\nMatches per decade:")
decade = (matches["date"].dt.year // 10 * 10).rename("decade")
decade_counts = decade.value_counts().sort_index()
for d, count in decade_counts.items():
    print(f"  {d}s: {count}")

# Task 4: Engineer features for the model

In [ ]:
from collections import defaultdict

MAJOR_TOURNAMENTS = {
    "Soccer World Cup",
    "Soccer World Cup qualification",
    "UEFA Euro",
    "UEFA Euro qualification",
    "Copa América",
    "African Cup of Nations",
}

# Helper functions — operate on a list of (goals_for, goals_against, won) tuples
def winrate(hist):
    return sum(h[2] for h in hist) / len(hist) if hist else 0.5

def goal_avg(hist):
    return sum(h[0] for h in hist) / len(hist) if hist else 1.0

def recent_form(hist):
    last10 = hist[-10:]
    return sum(h[2] for h in last10) / 10 if len(last10) == 10 else 0.5

# Filter to 1990-onwards and sort chronologically
filtered = (
    matches[matches["date"] >= pd.Timestamp("1990-01-01")]
    .sort_values(by="date")
    .reset_index(drop=True)
)

team_history = defaultdict(list)  # team -> [(goals_for, goals_against, won), ...]
rows = []

for _, row in filtered.iterrows():
    home, away = row["home_team"], row["away_team"]
    h_hist = team_history[home]
    a_hist = team_history[away]

    # --- Compute features from history BEFORE this match ---
    feat = {
        "date": row["date"],
        "home_team": home,
        "away_team": away,
        "team_a_winrate": winrate(h_hist),
        "team_b_winrate": winrate(a_hist),
        "team_a_goal_avg": goal_avg(h_hist),
        "team_b_goal_avg": goal_avg(a_hist),
        "team_a_recent_form": recent_form(h_hist),
        "team_b_recent_form": recent_form(a_hist),
        "is_neutral": int(row["neutral"]),
        "is_major_tournament": int(row["tournament"] in MAJOR_TOURNAMENTS),
    }

    # Outcome: 0 = home win, 1 = draw, 2 = away win
    hs, as_ = row["home_score"], row["away_score"]
    feat["outcome"] = 0 if hs > as_ else (1 if hs == as_ else 2)
    rows.append(feat)

    # --- Update history AFTER computing features (no leakage) ---
    home_won = int(hs > as_)
    away_won = int(as_ > hs)
    team_history[home].append((hs, as_, home_won))
    team_history[away].append((as_, hs, away_won))

features_df = pd.DataFrame(rows)

print(features_df.shape)
features_df.head(3)

# Task 5: Split data into training and test sets

In [ ]:
feature_cols = [
    "team_a_winrate",
    "team_b_winrate",
    "team_a_goal_avg",
    "team_b_goal_avg",
    "team_a_recent_form",
    "team_b_recent_form",
    "is_neutral",
    "is_major_tournament",
]

# Time-based split: train < 2018-01-01, test >= 2018-01-01
cutoff = pd.Timestamp("2018-01-01")
train_df = features_df[features_df["date"] < cutoff]
test_df  = features_df[features_df["date"] >= cutoff]

X_train = train_df[feature_cols]
X_test  = test_df[feature_cols]
y_train = train_df["outcome"]
y_test  = test_df["outcome"]

print("X_train shape:", X_train.shape)
print("X_test  shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test  shape:", y_test.shape)
print()
print("y_train class distribution:")
print(pd.Series(y_train).value_counts(normalize=True).sort_index().round(3).to_string())

## Task 6: Train and evaluate the model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import numpy as np

# Train
model = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Test accuracy
acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {acc * 100:.2f}%")

# Baseline: always predict the most frequent class in y_train
most_frequent = np.bincount(y_train).argmax()
baseline_acc = accuracy_score(y_test, np.full(len(y_test), most_frequent))
print(f"Baseline accuracy (most frequent class): {baseline_acc * 100:.2f}%")

# Confusion matrix
labels = [0, 1, 2]
label_names = ["Home win", "Draw", "Away win"]
cm = confusion_matrix(y_test, y_pred, labels=labels)
header = f"{'':12s}" + "".join(f"{n:>12s}" for n in label_names)
print("\nConfusion matrix (rows=actual, cols=predicted):")
print(header)
for row_name, row in zip(label_names, cm):
    print(f"{row_name:12s}" + "".join(f"{v:12d}" for v in row))

# Feature importances
print("\nFeature importances (descending):")
importances = model.feature_importances_
sorted_idx = np.argsort(importances)[::-1]
for i in sorted_idx:
    print(f"  {feature_cols[i]:35s} {importances[i]:.4f}")